<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/marco-canas/fundamentos_de_programacion/blob/main/2_clases/unidad2/5_chapter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/marco-canas/fundamentos_de_programacion/blob/main/2_clases/unidad2/5_chapter.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Ciclos `for` y `while` en Python
## Aplicados a Ciencia de Datos con NumPy y Pandas



En este notebook aprenderás:

1. La sintaxis y lógica de los ciclos `for` y `while`.
2. Cómo recorrer listas, arreglos de NumPy y DataFrames de Pandas.
3. Por qué en ciencia de datos casi siempre preferimos la **vectorización** en lugar de los ciclos explícitos, y cómo comparar el rendimiento.
4. Buenas prácticas y ejercicios para practicar.

>  Ejecuta las celdas en orden (`Shift + Enter`) para que las variables se vayan definiendo correctamente.


In [ ]:
# Librerías que usaremos a lo largo del notebook
import numpy as np
import pandas as pd
import time

print("NumPy versión:", np.__version__)
print("Pandas versión:", pd.__version__)


NumPy versión: 2.4.4
Pandas versión: 3.0.2


---
## 1. El ciclo `for`

El ciclo `for` se usa cuando **ya sabemos sobre qué colección vamos a iterar** (una lista, una tupla, un rango de números, las filas de un DataFrame, etc.). En cada vuelta, la variable de control toma un valor distinto del objeto iterable.

```python
for variable in iterable:
    # bloque de código que se repite
```


In [2]:
# Ejemplo básico: recorrer una lista de nombres de columnas
columnas = ["edad", "ingresos", "gasto_mensual", "ciudad"]

for col in columnas:
    print(f"Procesando columna: {col}")


Procesando columna: edad
Procesando columna: ingresos
Procesando columna: gasto_mensual
Procesando columna: ciudad


In [3]:
# Ejemplo con range(): sumar los primeros 10 números naturales
total = 0
for i in range(1, 11):
    total += i

print("La suma de 1 a 10 es:", total)


La suma de 1 a 10 es: 55


In [4]:
# for con enumerate(): útil cuando necesitamos el índice y el valor
notas = [7.5, 8.9, 6.2, 9.0]

for indice, nota in enumerate(notas):
    print(f"Estudiante {indice}: nota = {nota}")


Estudiante 0: nota = 7.5
Estudiante 1: nota = 8.9
Estudiante 2: nota = 6.2
Estudiante 3: nota = 9.0


### `for` con `break` y `continue`

- `break` interrumpe el ciclo por completo.
- `continue` salta a la siguiente iteración sin ejecutar el resto del bloque.


In [5]:
ventas_diarias = [120, 340, 0, 500, -20, 610]

for venta in ventas_diarias:
    if venta < 0:
        print("Valor inválido detectado, deteniendo el proceso.")
        break
    if venta == 0:
        continue  # saltamos los días sin ventas
    print(f"Venta procesada: {venta}")


Venta procesada: 120
Venta procesada: 340
Venta procesada: 500
Valor inválido detectado, deteniendo el proceso.


---
## 2. El ciclo `while`

El ciclo `while` se usa cuando **no sabemos de antemano cuántas veces** se repetirá el bloque: se repite **mientras** se cumpla una condición.

```python
while condicion:
    # bloque de código
```

⚠️ Cuidado: si la condición nunca se vuelve falsa, obtenemos un ciclo infinito. Siempre debe existir algo dentro del ciclo que eventualmente cambie la condición.


In [6]:
# Ejemplo: simular un proceso iterativo hasta alcanzar una meta de ahorro
ahorro = 0
meta = 1000
mes = 0
aporte_mensual = 150

while ahorro < meta:
    ahorro += aporte_mensual
    mes += 1
    print(f"Mes {mes}: ahorro acumulado = {ahorro}")

print(f"\n Meta alcanzada en el mes {mes} con un ahorro de {ahorro}")


Mes 1: ahorro acumulado = 150
Mes 2: ahorro acumulado = 300
Mes 3: ahorro acumulado = 450
Mes 4: ahorro acumulado = 600
Mes 5: ahorro acumulado = 750
Mes 6: ahorro acumulado = 900
Mes 7: ahorro acumulado = 1050

 Meta alcanzada en el mes 7 con un ahorro de 1050


In [7]:
# Ejemplo típico en ciencia de datos: iterar hasta que un algoritmo "converja"
valor_actual = 100.0
tolerancia = 0.01
factor = 0.5
iteracion = 0

while valor_actual > tolerancia:
    valor_actual = valor_actual * factor  # el valor se reduce a la mitad cada vez
    iteracion += 1

print(f"Convergió en {iteracion} iteraciones, valor final = {valor_actual:.5f}")


Convergió en 14 iteraciones, valor final = 0.00610


---
## 3. De listas puras a NumPy

Antes de trabajar con NumPy, así es como calcularíamos operaciones elemento por elemento usando **solo Python puro** con ciclos `for`. Esto nos servirá de punto de comparación.


In [8]:
precios = [10.5, 20.0, 35.75, 8.2, 100.0]
iva = 0.19

precios_con_iva = []
for precio in precios:
    precios_con_iva.append(precio * (1 + iva))

print(precios_con_iva)


[12.495, 23.799999999999997, 42.5425, 9.758, 119.0]


---
## 4. Ciclos con NumPy: ¿por qué evitarlos cuando podemos?

NumPy está construido para trabajar con **operaciones vectorizadas**: aplicar una operación a todo un arreglo de una sola vez, sin necesidad de un ciclo `for` explícito en Python. Internamente, NumPy sí itera, pero lo hace en código C compilado, mucho más rápido que un ciclo en Python puro.

Veamos la diferencia con un ejemplo y midamos el tiempo de ejecución.


In [9]:
# Creamos un arreglo grande de números aleatorios
np.random.seed(42)
n = 1_000_000
datos = np.random.rand(n) * 100
print(datos[:5])


[37.45401188 95.07143064 73.19939418 59.86584842 15.60186404]


In [10]:
# Opción 1: usar un ciclo for para calcular el cuadrado de cada elemento
inicio = time.time()

resultado_for = np.empty(n)
for i in range(n):
    resultado_for[i] = datos[i] ** 2

tiempo_for = time.time() - inicio
print(f"Tiempo con ciclo for: {tiempo_for:.4f} segundos")


Tiempo con ciclo for: 0.2545 segundos


In [11]:
# Opción 2: usar vectorización de NumPy (sin ciclo explícito en Python)
inicio = time.time()

resultado_vectorizado = datos ** 2

tiempo_vectorizado = time.time() - inicio
print(f"Tiempo vectorizado: {tiempo_vectorizado:.6f} segundos")

print(f"\nNumPy vectorizado fue aproximadamente {tiempo_for / tiempo_vectorizado:.0f} veces más rápido")


Tiempo vectorizado: 0.031055 segundos

NumPy vectorizado fue aproximadamente 8 veces más rápido


**Conclusión:** cuando trabajamos con arreglos de NumPy, casi siempre existe una forma vectorizada (`+`, `-`, `*`, `/`, `np.sqrt`, `np.where`, `np.sum`, comparaciones booleanas, etc.) que reemplaza al ciclo `for`. Reserva los ciclos para NumPy solo cuando:

- La lógica es tan compleja que no existe una función vectorizada equivalente.
- Necesitas depender de resultados de iteraciones anteriores de forma no vectorizable (por ejemplo, ciertos procesos recursivos).


In [12]:
# Ejemplo de "for aceptable" en NumPy: una serie acumulativa dependiente del paso anterior
n_pasos = 10
serie = np.zeros(n_pasos)
serie[0] = 1.0

for t in range(1, n_pasos):
    serie[t] = serie[t-1] * 1.05 + 2  # cada valor depende del anterior

print(serie)


[ 1.          3.05        5.2025      7.462625    9.83575625 12.32754406
 14.94392127 17.69111733 20.5756732  23.60445686]


---
## 5. Ciclos con Pandas: `iterrows`, `itertuples`, `apply` y la alternativa vectorizada

En Pandas es muy común la tentación de recorrer un DataFrame fila por fila con un ciclo `for`. Existen varias formas de hacerlo, ordenadas de **más lenta a más rápida**:

1. `for ... in df.iterrows()` → más lenta, devuelve cada fila como una `Series`.
2. `for ... in df.itertuples()` → más rápida que `iterrows`, devuelve tuplas.
3. `df.apply(funcion, axis=1)` → más legible, pero internamente sigue siendo un ciclo.
4. **Operaciones vectorizadas / booleanas** → la opción más rápida y "pandonica".


In [13]:
# Creamos un DataFrame de ejemplo: ventas de una tienda
df = pd.DataFrame({
    "producto": ["A", "B", "C", "D", "E"],
    "precio": [10.0, 25.5, 7.8, 50.0, 15.3],
    "cantidad": [3, 1, 10, 2, 5]
})
df


,producto,precio,cantidad
0,A,10.0,3
1,B,25.5,1
2,C,7.8,10
3,D,50.0,2
4,E,15.3,5


In [14]:
# Opción 1 (lenta): iterrows()
totales = []
for indice, fila in df.iterrows():
    totales.append(fila["precio"] * fila["cantidad"])

df["total_iterrows"] = totales
df


,producto,precio,cantidad,total_iterrows
0,A,10.0,3,30.0
1,B,25.5,1,25.5
2,C,7.8,10,78.0
3,D,50.0,2,100.0
4,E,15.3,5,76.5


In [15]:
# Opción 2 (más rápida): itertuples()
totales = []
for fila in df.itertuples():
    totales.append(fila.precio * fila.cantidad)

df["total_itertuples"] = totales
df


,producto,precio,cantidad,total_iterrows,total_itertuples
0,A,10.0,3,30.0,30.0
1,B,25.5,1,25.5,25.5
2,C,7.8,10,78.0,78.0
3,D,50.0,2,100.0,100.0
4,E,15.3,5,76.5,76.5


In [16]:
# Opción 3: apply() con axis=1
df["total_apply"] = df.apply(lambda fila: fila["precio"] * fila["cantidad"], axis=1)
df


,producto,precio,cantidad,total_iterrows,total_itertuples,total_apply
0,A,10.0,3,30.0,30.0,30.0
1,B,25.5,1,25.5,25.5,25.5
2,C,7.8,10,78.0,78.0,78.0
3,D,50.0,2,100.0,100.0,100.0
4,E,15.3,5,76.5,76.5,76.5


In [17]:
# Opción 4 (recomendada): operación vectorizada directa sobre las columnas
df["total_vectorizado"] = df["precio"] * df["cantidad"]
df


,producto,precio,cantidad,total_iterrows,total_itertuples,total_apply,total_vectorizado
0,A,10.0,3,30.0,30.0,30.0,30.0
1,B,25.5,1,25.5,25.5,25.5,25.5
2,C,7.8,10,78.0,78.0,78.0,78.0
3,D,50.0,2,100.0,100.0,100.0,100.0
4,E,15.3,5,76.5,76.5,76.5,76.5


Como puedes ver, las cuatro columnas (`total_iterrows`, `total_itertuples`, `total_apply`, `total_vectorizado`) dan el mismo resultado, pero la última se escribe en **una sola línea**, es más legible y es la más eficiente.

Comparemos el tiempo con un DataFrame más grande.


In [18]:
# Comparación de rendimiento con un DataFrame grande
np.random.seed(0)
n_filas = 100_000
df_grande = pd.DataFrame({
    "precio": np.random.rand(n_filas) * 100,
    "cantidad": np.random.randint(1, 20, size=n_filas)
})

# --- iterrows ---
inicio = time.time()
totales = []
for indice, fila in df_grande.iterrows():
    totales.append(fila["precio"] * fila["cantidad"])
tiempo_iterrows = time.time() - inicio

# --- vectorizado ---
inicio = time.time()
totales_vect = df_grande["precio"] * df_grande["cantidad"]
tiempo_vectorizado = time.time() - inicio

print(f"iterrows():      {tiempo_iterrows:.4f} s")
print(f"vectorizado:     {tiempo_vectorizado:.6f} s")
print(f"Vectorizado fue ~{tiempo_iterrows/tiempo_vectorizado:.0f} veces más rápido")


iterrows():      1.6863 s
vectorizado:     0.000738 s
Vectorizado fue ~2285 veces más rápido


### ¿Cuándo SÍ usar un `for` con Pandas?

- Cuando necesitas aplicar lógica muy compleja, con condicionales anidados difíciles de vectorizar.
- Cuando estás generando reportes o exportando archivo por archivo/fila (efectos secundarios, no cálculos puros).
- Al prototipar rápido con DataFrames pequeños, antes de optimizar.


In [19]:
# Ejemplo razonable de for en Pandas: generar un mensaje personalizado por fila (efecto secundario, no cálculo numérico)
for fila in df.itertuples():
    print(f"El producto {fila.producto} generó ${fila.total_vectorizado:.2f} en ventas.")


El producto A generó $30.00 en ventas.
El producto B generó $25.50 en ventas.
El producto C generó $78.00 en ventas.
El producto D generó $100.00 en ventas.
El producto E generó $76.50 en ventas.


In [20]:
# while en Pandas: ir agregando filas hasta cumplir una condición de negocio
carrito = pd.DataFrame(columns=["producto", "precio"])
productos_disponibles = [("Mouse", 15.0), ("Teclado", 25.0), ("Monitor", 120.0), ("Silla", 80.0)]

presupuesto = 150
i = 0
gasto_total = 0

while gasto_total < presupuesto and i < len(productos_disponibles):
    nombre, precio = productos_disponibles[i]
    if gasto_total + precio <= presupuesto:
        carrito.loc[len(carrito)] = [nombre, precio]
        gasto_total += precio
    i += 1

print(f"Gasto total: ${gasto_total}")
carrito


Gasto total: $120.0


,producto,precio
0,Mouse,15.0
1,Teclado,25.0
2,Silla,80.0


---
## 6. Resumen y buenas prácticas

| Situación | Recomendación |
|---|---|
| Iterar sobre listas/diccionarios normales de Python | `for` es perfectamente adecuado |
| No sabes cuántas iteraciones necesitas (esperar una condición) | `while` |
| Operaciones matemáticas sobre arreglos NumPy | Vectorización (`+`, `*`, `np.where`, etc.) en vez de `for` |
| Cálculos columna a columna en Pandas | Operaciones vectorizadas sobre columnas (`df["a"] + df["b"]`) |
| Necesitas recorrer filas de un DataFrame sí o sí | Prefiere `itertuples()` sobre `iterrows()` |
| Lógica compleja no vectorizable | `apply()` o un `for` explícito, aceptando el costo en rendimiento |

**Regla de oro:** en ciencia de datos, antes de escribir un `for` sobre un arreglo de NumPy o un DataFrame de Pandas, pregúntate: *"¿existe una función u operación vectorizada que haga esto mismo?"*


---
## 7. Ejercicios propuestos

Resuelve cada ejercicio en la celda de código correspondiente.

**Ejercicio 1.** Usando un ciclo `for`, crea una lista con los cuadrados de los números del 1 al 20 que sean pares.

**Ejercicio 2.** Usando un ciclo `while`, encuentra el primer número de Fibonacci mayor a 1000.

**Ejercicio 3.** Dado el arreglo de NumPy `edades = np.array([15, 22, 35, 44, 12, 67, 18, 9])`, calcula cuántas personas son mayores de edad (>=18) **sin usar un ciclo for**, usando operaciones booleanas de NumPy.

**Ejercicio 4.** Dado el DataFrame `df` de la sección 5, agrega una columna `categoria` que diga `"alto"` si `total_vectorizado > 50` y `"bajo"` en caso contrario, usando `np.where` (vectorizado) y comparando el resultado con lo que obtendrías usando `apply()`.


In [21]:
# Ejercicio 1: tu código aquí



In [22]:
# Ejercicio 2: tu código aquí



In [23]:
# Ejercicio 3: tu código aquí
edades = np.array([15, 22, 35, 44, 12, 67, 18, 9])



In [24]:
# Ejercicio 4: tu código aquí



---
### ¡Felicidades! 🎉

Ya conoces cómo usar `for` y `while` en Python, y sobre todo, **cuándo NO usarlos** en NumPy y Pandas a favor de la vectorización, la herramienta clave para escribir código de ciencia de datos eficiente.
